# Preparación de Datos - Train/Test Split

**Proyecto:** Sistema de Clasificación de Acciones S&P 500

**Grupo 27** - Universidad de Los Andes

---

## Objetivo

Preparar los datos para modelado:
- Cargar dataset consolidado
- Split temporal train/test (80/20)
- Guardar archivos separados para reutilizar en todos los experimentos

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

## 1. Carga de Datos

In [2]:
# Cargar dataset completo
df = pd.read_parquet('../../data/processed/ml_ready/features_combined.parquet')

print(f"Dataset cargado: {df.shape[0]:,} filas, {df.shape[1]} columnas")
df.head()

Dataset cargado: 25,160 filas, 25 columnas


,Ticker,Date,Open,High,Low,Close,Volume,SMA_20,SMA_50,EMA_12,...,BB_upper,BB_middle,BB_lower,BB_width,ATR_14,OBV,Returns,Volatility_10,Volume_change,Target
0,AAPL,2015-01-02 00:00:00-05:00,24.694239,24.705324,23.798604,24.237555,212818400,24.793442,24.727228,24.769349,...,25.776754,24.793442,23.810131,0.079320,0.547143,219697360800,-0.009512,0.014881,0.285030,0
1,AAPL,2015-01-05 00:00:00-05:00,24.006992,24.086801,23.368521,23.554741,257142000,24.691021,24.743654,24.582487,...,25.740156,24.691021,23.641886,0.084981,0.570135,219440218800,-0.028172,0.013245,0.208270,1
2,AAPL,2015-01-06 00:00:00-05:00,23.619033,23.816338,23.195601,23.556959,263188400,24.594142,24.752001,24.424713,...,25.685577,24.594142,23.502706,0.088756,0.573750,219703407200,0.000094,0.013346,0.023514,1
3,AAPL,2015-01-07 00:00:00-05:00,23.765356,23.987048,23.654510,23.887287,160423600,24.542599,24.765233,24.342032,...,25.664890,24.542599,23.420307,0.091457,0.563488,219863830800,0.014023,0.013852,-0.390461,1
4,AAPL,2015-01-08 00:00:00-05:00,24.215380,24.862719,24.097882,24.805079,237458000,24.517880,24.797306,24.413270,...,25.593198,24.517880,23.442563,0.087717,0.592913,220101288800,0.038422,0.019440,0.480194,1


In [3]:
# Verificar estructura
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25160 entries, 0 to 25159
Data columns (total 25 columns):
 #   Column         Non-Null Count  Dtype                           
---  ------         --------------  -----                           
 0   Ticker         25160 non-null  object                          
 1   Date           25160 non-null  datetime64[ns, America/New_York]
 2   Open           25160 non-null  float64                         
 3   High           25160 non-null  float64                         
 4   Low            25160 non-null  float64                         
 5   Close          25160 non-null  float64                         
 6   Volume         25160 non-null  int64                           
 7   SMA_20         25160 non-null  float64                         
 8   SMA_50         25160 non-null  float64                         
 9   EMA_12         25160 non-null  float64                         
 10  EMA_26         25160 non-null  float64                    

In [4]:
# Distribución de Target
print("\nDistribución de Target:")
print(df['Target'].value_counts())
print(f"\nBalance: {df['Target'].value_counts(normalize=True) * 100}")


Distribución de Target:
Target
1    13281
0    11879
Name: count, dtype: int64

Balance: Target
1    52.786169
0    47.213831
Name: proportion, dtype: float64


## 2. Train/Test Split Temporal

Para datos financieros es crítico usar split temporal:
- Entrenar con datos pasados
- Evaluar en datos futuros (simulando producción)

**Split:** 80% train / 20% test

In [5]:
# Ordenar por fecha para asegurar split temporal correcto
df_sorted = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

# Calcular índice de corte (80%)
split_idx = int(len(df_sorted) * 0.8)

# Separar train y test
train = df_sorted.iloc[:split_idx].copy()
test = df_sorted.iloc[split_idx:].copy()

print(f"Train set: {len(train):,} filas ({len(train)/len(df_sorted)*100:.1f}%)")
print(f"Test set:  {len(test):,} filas ({len(test)/len(df_sorted)*100:.1f}%)")

Train set: 20,128 filas (80.0%)
Test set:  5,032 filas (20.0%)


In [6]:
# Verificar rangos de fechas
print("\nRangos de fechas:")
print(f"Train: {train['Date'].min()} a {train['Date'].max()}")
print(f"Test:  {test['Date'].min()} a {test['Date'].max()}")


Rangos de fechas:
Train: 2015-01-02 00:00:00-05:00 a 2024-12-31 00:00:00-05:00
Test:  2015-01-02 00:00:00-05:00 a 2024-12-31 00:00:00-05:00


In [7]:
# Verificar balance en ambos sets
print("\nBalance de Target en Train:")
print(train['Target'].value_counts(normalize=True) * 100)

print("\nBalance de Target en Test:")
print(test['Target'].value_counts(normalize=True) * 100)


Balance de Target en Train:
Target
1    53.07035
0    46.92965
Name: proportion, dtype: float64

Balance de Target en Test:
Target
1    51.649444
0    48.350556
Name: proportion, dtype: float64


## 3. Guardar Datasets

Guardamos train.parquet y test.parquet para reutilizar en todos los notebooks de modelado.

In [8]:
# Guardar datasets procesados
train.to_parquet('../../data/processed/ml_ready/train.parquet', index=False)
test.to_parquet('../../data/processed/ml_ready/test.parquet', index=False)

print("Datasets guardados:")
print("  data/processed/ml_ready/train.parquet")
print("  data/processed/ml_ready/test.parquet")

Datasets guardados:
  data/processed/ml_ready/train.parquet
  data/processed/ml_ready/test.parquet


## Resumen

- Dataset original: 25,160 observaciones
- Train: 20,128 observaciones (2015 a ~2023)
- Test: 5,032 observaciones (~2023 a 2024)
- Balance mantenido en ambos sets

**Próximos pasos:** Usar estos datasets en los notebooks de modelado.